# Spatial Conflict Detection

Interactive analysis of spatial overlaps between construction permits and inspection locations.

In [ ]:
import sys
sys.path.insert(0, '/home/user/nyc_data')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import folium
from folium import plugins
import ipywidgets as widgets

print("✓ Environment initialized")

In [ ]:
def generate_sample_spatial_data():
    """Generate sample spatial data for demonstration."""
    np.random.seed(42)
    
    # NYC borough centers (lat, lon)
    borough_coords = {
        'MANHATTAN': (40.7831, -73.9712),
        'BRONX': (40.8448, -73.8648),
        'BROOKLYN': (40.6782, -73.9442),
        'QUEENS': (40.7282, -73.7949),
        'STATEN_ISLAND': (40.5795, -74.1502)
    }
    
    # Generate points around borough centers
    permits = []
    inspections = []
    
    for borough, (lat, lon) in borough_coords.items():
        # Permits
        for _ in range(20):
            permits.append({
                'borough': borough,
                'lat': lat + np.random.normal(0, 0.02),
                'lon': lon + np.random.normal(0, 0.02),
                'type': 'permit',
                'cost': np.random.uniform(10000, 500000)
            })
        
        # Inspections
        for _ in range(30):
            inspections.append({
                'borough': borough,
                'lat': lat + np.random.normal(0, 0.02),
                'lon': lon + np.random.normal(0, 0.02),
                'type': 'inspection',
                'violations': np.random.poisson(2)
            })
    
    return pd.DataFrame(permits), pd.DataFrame(inspections)

permits, inspections = generate_sample_spatial_data()

print(f"✓ Generated {len(permits)} permits and {len(inspections)} inspection locations")

## Interactive Map

In [ ]:
# Create map centered on NYC
m = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=10,
    tiles='OpenStreetMap'
)

# Add permits (blue)
for idx, row in permits.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        popup=f"Permit - {row['borough']}",
        color='blue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.6
    ).add_to(m)

# Add inspections (red)
for idx, row in inspections.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        popup=f"Inspection - {row['borough']}",
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.5
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 180px; height: 100px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:14px; padding: 10px">
     <p style="margin: 0 0 10px 0;"><b>Legend</b></p>
     <p style="margin: 5px 0;"><i class="fa fa-circle" style="color:blue"></i> Permits</p>
     <p style="margin: 5px 0;"><i class="fa fa-circle" style="color:red"></i> Inspections</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m

## Conflict Analysis

In [ ]:
# Count spatial overlaps by borough
borough_conflicts = pd.DataFrame({
    'borough': permits['borough'].unique(),
    'permits': [len(permits[permits['borough'] == b]) for b in permits['borough'].unique()],
    'inspections': [len(inspections[inspections['borough'] == b]) for b in permits['borough'].unique()]
})

# Estimate conflicts (simplified)
borough_conflicts['estimated_conflicts'] = np.minimum(
    borough_conflicts['permits'], 
    borough_conflicts['inspections']
) // 2

fig = px.bar(
    borough_conflicts,
    x='borough',
    y=['permits', 'inspections', 'estimated_conflicts'],
    title='Permits vs Inspections by Borough',
    barmode='group',
    labels={'value': 'Count', 'borough': 'Borough'},
    height=400
)

fig.show()

## Distance Analysis

In [ ]:
from math import radians, cos, sin, asin, sqrt

def haversine(lon1, lat1, lon2, lat2):
    """Calculate distance between two points (miles)."""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 3959
    return c * r

# Calculate minimum distances
min_distances = []
for _, permit in permits.iterrows():
    distances = [haversine(permit['lon'], permit['lat'], insp['lon'], insp['lat'])
                 for _, insp in inspections.iterrows()]
    min_distances.append(min(distances))

fig = px.histogram(
    x=min_distances,
    nbins=20,
    title='Distance from Permits to Nearest Inspection',
    labels={'x': 'Distance (miles)', 'y': 'Count'},
    height=400
)

fig.show()

print(f"Mean minimum distance: {np.mean(min_distances):.3f} miles")
print(f"Median minimum distance: {np.median(min_distances):.3f} miles")
print(f"Max minimum distance: {np.max(min_distances):.3f} miles")